In [1]:
import rasterio
import numpy as np
import geopandas as gpd
from shapely.geometry import shape, box
from rasterio.features import shapes
import matplotlib.pyplot as plt
import joblib
import os
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Load model terbaik dari Fase 2
model_path = '02_Modeling/best_model_rf_lahan.pkl'
model = joblib.load(model_path)

# KODE AJAIB: Otomatis deteksi jumlah band yang dipakai model (6 atau 11)
num_bands_model = model.n_features_in_ 
print(f"✅ Model loaded. Model ini dilatih dengan {num_bands_model} band fitur.")

✅ Model loaded. Model ini dilatih dengan 6 band fitur.


In [2]:
def classify_and_save(raster_in, model, raster_out, num_bands):
    print(f"Memproses: {raster_in}")
    with rasterio.open(raster_in) as src:
        img = src.read()
        meta = src.meta.copy()
        
        # Reshape ke (pixels, bands)
        bands_count, rows, cols = img.shape
        X_img = img.reshape(bands_count, -1).T
        
        # Otomatis potong array sesuai jumlah fitur model
        X_img = X_img[:, :num_bands]
        
        # Handle NaN/NoData
        valid_mask = ~np.isnan(X_img).any(axis=1)
        X_valid = X_img[valid_mask]
        
        # Prediksi
        pred_valid = model.predict(X_valid)
        
        # Reconstruct
        pred_full = np.full(X_img.shape[0], 255, dtype='uint8') # 255 = NoData
        pred_full[valid_mask] = pred_valid
        pred_2d = pred_full.reshape(rows, cols)
        
        # Save Raster
        meta.update(dtype='uint8', count=1, compress='lzw', nodata=255)
        with rasterio.open(raster_out, 'w', **meta) as dst:
            dst.write(pred_2d, 1)
            
    print(f"✅ Tersimpan: {raster_out}")
    return pred_2d, src.transform

os.makedirs('03_Hasil_Klasifikasi', exist_ok=True)

pred_2024, transform_2024 = classify_and_save(
    'raw/feature_stack_2024.tif', model, 
    '03_Hasil_Klasifikasi/klasifikasi_lahan_2024.tif', num_bands_model
)

pred_2025, transform_2025 = classify_and_save(
    'raw/feature_stack_2025.tif', model, 
    '03_Hasil_Klasifikasi/klasifikasi_lahan_2025.tif', num_bands_model
)

Memproses: raw/feature_stack_2024.tif
✅ Tersimpan: 03_Hasil_Klasifikasi/klasifikasi_lahan_2024.tif
Memproses: raw/feature_stack_2025.tif
✅ Tersimpan: 03_Hasil_Klasifikasi/klasifikasi_lahan_2025.tif


In [3]:
def calculate_area_hectares(raster_path):
    with rasterio.open(raster_path) as src:
        data = src.read(1)
        # Hitung pixel target (nilai = 1), abaikan NoData (255)
        target_pixels = np.sum(data == 1)
        
        # Buat bounding box untuk konversi CRS
        bbox = box(*src.bounds) 
        gdf_temp = gpd.GeoDataFrame({'geometry': [bbox]}, crs=src.crs)
        
        # Konversi CRS ke UTM 51S untuk hitung luas akurat (Morowali)
        gdf_utm = gdf_temp.to_crs(epsg=32751) 
        
        # Hitung luas 1 pixel dalam meter persegi
        pixel_width_m = (gdf_utm.bounds.maxx.iloc[0] - gdf_utm.bounds.minx.iloc[0]) / src.width
        pixel_height_m = (gdf_utm.bounds.maxy.iloc[0] - gdf_utm.bounds.miny.iloc[0]) / src.height
        pixel_area_m2 = pixel_width_m * pixel_height_m
        
        total_area_ha = (target_pixels * pixel_area_m2) / 10000
        return total_area_ha, pixel_area_m2

print("Menghitung luas area...")
luas_2024, pixel_area_m2_2024 = calculate_area_hectares('03_Hasil_Klasifikasi/klasifikasi_lahan_2024.tif')
luas_2025, pixel_area_m2_2025 = calculate_area_hectares('03_Hasil_Klasifikasi/klasifikasi_lahan_2025.tif')

pixel_area_m2 = (pixel_area_m2_2024 + pixel_area_m2_2025) / 2 

print(f"\n📊 HASIL PERHITUNGAN LUAS:")
print(f"Luas Lahan Terbuka 2024: {luas_2024:,.2f} Hektar")
print(f"Luas Lahan Terbuka 2025: {luas_2025:,.2f} Hektar")
print(f"Perubahan Bersih: {luas_2025 - luas_2024:+,.2f} Hektar")

Menghitung luas area...

📊 HASIL PERHITUNGAN LUAS:
Luas Lahan Terbuka 2024: 21,109.43 Hektar
Luas Lahan Terbuka 2025: 19,926.84 Hektar
Perubahan Bersih: -1,182.59 Hektar


In [4]:
# Buat Change Map: 0=Stable Non-Target, 1=Loss (1->0), 2=Gain (0->1), 3=Stable Target
valid_2024 = (pred_2024 != 255)
valid_2025 = (pred_2025 != 255)
valid_both = valid_2024 & valid_2025

change_map = np.full(pred_2024.shape, 255, dtype='uint8')
change_map[valid_both] = (pred_2025[valid_both] * 2) + pred_2024[valid_both]

# Hitung statistik perubahan
unique, counts = np.unique(change_map[valid_both], return_counts=True)
stats = dict(zip(unique, counts))

pixel_area_ha = pixel_area_m2 / 10000

print("\n📈 STATISTIK PERUBAHAN:")
print(f"0 (Stable Non-Target): {stats.get(0, 0) * pixel_area_ha:,.2f} Ha")
print(f"1 (Loss / Berkurang)   : {stats.get(1, 0) * pixel_area_ha:,.2f} Ha")
print(f"2 (Gain / Bertambah)   : {stats.get(2, 0) * pixel_area_ha:,.2f} Ha")
print(f"3 (Stable Target)      : {stats.get(3, 0) * pixel_area_ha:,.2f} Ha")

# Simpan Change Map
with rasterio.open('03_Hasil_Klasifikasi/change_map.tif', 'w', driver='GTiff',
                   height=pred_2024.shape[0], width=pred_2024.shape[1],
                   count=1, dtype='uint8', crs='EPSG:4326', transform=transform_2024,
                   nodata=255, compress='lzw') as dst:
    dst.write(change_map, 1)
print("\n✅ Change map tersimpan.")


📈 STATISTIK PERUBAHAN:
0 (Stable Non-Target): 56,699.38 Ha
1 (Loss / Berkurang)   : 7,962.14 Ha
2 (Gain / Bertambah)   : 2,182.46 Ha
3 (Stable Target)      : 13,145.11 Ha

✅ Change map tersimpan.


In [5]:
def raster_to_geojson(raster_path, class_value, output_path, min_area_ha=1.0):
    print(f"Vectorizing class {class_value} from {raster_path}...")
    with rasterio.open(raster_path) as src:
        data = src.read(1)
        mask = (data == class_value)
        
        shapes_gen = shapes(data, mask=mask, transform=src.transform)
        geoms = [{'geometry': shape(s), 'properties': {'class': class_value}} 
                 for s, v in shapes_gen if v == class_value]
        
    gdf = gpd.GeoDataFrame.from_features(geoms, crs=src.crs)
    
    if len(gdf) == 0:
        print("No features found.")
        return
        
    # Konversi ke UTM untuk hitung luas & filter patch kecil
    gdf_utm = gdf.to_crs(epsg=32751)
    gdf_utm['area_ha'] = gdf_utm.geometry.area / 10000
    
    # Filter patch < min_area_ha (Hapus noise pixel kecil)
    gdf_filtered = gdf_utm[gdf_utm['area_ha'] >= min_area_ha].copy()
    
    # Simplify geometry agar file GeoJSON tidak terlalu berat untuk WebGIS
    gdf_filtered['geometry'] = gdf_filtered.geometry.simplify(0.0001)
    
    # Kembali ke EPSG:4326 untuk GeoJSON
    gdf_final = gdf_filtered.to_crs(epsg=4326)
    gdf_final.to_file(output_path, driver='GeoJSON')
    print(f"✅ Tersimpan {len(gdf_final)} poligon ke {output_path}")

os.makedirs('04_GeoJSON_WebGIS', exist_ok=True)

# Export Target 2024, Target 2025, Gain, dan Loss
raster_to_geojson('03_Hasil_Klasifikasi/klasifikasi_lahan_2024.tif', 1, '04_GeoJSON_WebGIS/target_2024.geojson')
raster_to_geojson('03_Hasil_Klasifikasi/klasifikasi_lahan_2025.tif', 1, '04_GeoJSON_WebGIS/target_2025.geojson')
raster_to_geojson('03_Hasil_Klasifikasi/change_map.tif', 2, '04_GeoJSON_WebGIS/gain.geojson')
raster_to_geojson('03_Hasil_Klasifikasi/change_map.tif', 1, '04_GeoJSON_WebGIS/loss.geojson')

print("\n🎉 FASE 3 SELESAI! Semua raster dan GeoJSON siap diserahkan ke Anggota 4 (WebGIS).")

Vectorizing class 1 from 03_Hasil_Klasifikasi/klasifikasi_lahan_2024.tif...
✅ Tersimpan 926 poligon ke 04_GeoJSON_WebGIS/target_2024.geojson
Vectorizing class 1 from 03_Hasil_Klasifikasi/klasifikasi_lahan_2025.tif...
✅ Tersimpan 694 poligon ke 04_GeoJSON_WebGIS/target_2025.geojson
Vectorizing class 2 from 03_Hasil_Klasifikasi/change_map.tif...
✅ Tersimpan 377 poligon ke 04_GeoJSON_WebGIS/gain.geojson
Vectorizing class 1 from 03_Hasil_Klasifikasi/change_map.tif...
✅ Tersimpan 1083 poligon ke 04_GeoJSON_WebGIS/loss.geojson

🎉 FASE 3 SELESAI! Semua raster dan GeoJSON siap diserahkan ke Anggota 4 (WebGIS).


In [6]:
import pandas as pd

# Data dari Fase 3 (dalam Hektar)
stable_nontarget = 66718.48
loss = 3668.91
gain = 1450.37
stable_target = 8151.32

luas_kota_total = stable_nontarget + loss + gain + stable_target
luas_2024 = stable_target + loss
luas_2025 = stable_target + gain

# 1. Persentase terhadap Luas Kota (AOI)
pct_2024_kota = (luas_2024 / luas_kota_total) * 100
pct_2025_kota = (luas_2025 / luas_kota_total) * 100

# 2. Persentase Perubahan terhadap Luas Target 2024
pct_gain = (gain / luas_2024) * 100
pct_loss = (loss / luas_2024) * 100
pct_net_change = ((luas_2025 - luas_2024) / luas_2024) * 100

print("="*70)
print("📊 ANALISIS PERUBAHAN LAHAN TERBUKA (MOROWALI/BAHODOPI)")
print("="*70)

print(f"\n🌍 1. LUAS & PERSENTASE TERHADAP WILAYAH STUDI")
print(f"Total Luas Wilayah Studi (AOI): {luas_kota_total:,.2f} Ha")
print(f"Luas Lahan Terbuka 2024       : {luas_2024:,.2f} Ha ({pct_2024_kota:.2f}% dari AOI)")
print(f"Luas Lahan Terbuka 2025       : {luas_2025:,.2f} Ha ({pct_2025_kota:.2f}% dari AOI)")

print(f"\n📈 2. STATISTIK PERUBAHAN (TRANSISI 2024 -> 2025)")
print(f"Area Tetap Lahan Terbuka (Stable): {stable_target:,.2f} Ha")
print(f"Area Bertambah (Gain / 0->1)     : {gain:,.2f} Ha (+{pct_gain:.2f}% dr total 2024)")
print(f"Area Berkurang (Loss / 1->0)     : {loss:,.2f} Ha (-{pct_loss:.2f}% dr total 2024)")
print(f"Perubahan Bersih (Net Change)    : {luas_2025 - luas_2024:+,.2f} Ha ({pct_net_change:+.2f}% dr total 2024)")

print(f"\n🔄 3. MATRIKS TRANSISI (dalam Hektar)")
print(f"{'Dari \\ Ke':<20} | {'Non-Target 2025':<18} | {'Target 2025':<18} | {'Total 2024':<18}")
print("-" * 80)
print(f"{'Non-Target 2024':<20} | {stable_nontarget:<18,.2f} | {gain:<18,.2f} | {stable_nontarget+gain:<18,.2f}")
print(f"{'Target 2024':<20} | {loss:<18,.2f} | {stable_target:<18,.2f} | {loss+stable_target:<18,.2f}")
print("-" * 80)
print(f"{'Total 2025':<20} | {stable_nontarget+loss:<18,.2f} | {gain+stable_target:<18,.2f} | {luas_kota_total:<18,.2f}")

📊 ANALISIS PERUBAHAN LAHAN TERBUKA (MOROWALI/BAHODOPI)

🌍 1. LUAS & PERSENTASE TERHADAP WILAYAH STUDI
Total Luas Wilayah Studi (AOI): 79,989.08 Ha
Luas Lahan Terbuka 2024       : 11,820.23 Ha (14.78% dari AOI)
Luas Lahan Terbuka 2025       : 9,601.69 Ha (12.00% dari AOI)

📈 2. STATISTIK PERUBAHAN (TRANSISI 2024 -> 2025)
Area Tetap Lahan Terbuka (Stable): 8,151.32 Ha
Area Bertambah (Gain / 0->1)     : 1,450.37 Ha (+12.27% dr total 2024)
Area Berkurang (Loss / 1->0)     : 3,668.91 Ha (-31.04% dr total 2024)
Perubahan Bersih (Net Change)    : -2,218.54 Ha (-18.77% dr total 2024)

🔄 3. MATRIKS TRANSISI (dalam Hektar)
Dari \ Ke            | Non-Target 2025    | Target 2025        | Total 2024        
--------------------------------------------------------------------------------
Non-Target 2024      | 66,718.48          | 1,450.37           | 68,168.85         
Target 2024          | 3,668.91           | 8,151.32           | 11,820.23         
----------------------------------------------